In [5]:


import getpass
import json
import os
import re
import textwrap

from google import genai

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass(
        "Paste your Gemini API key. It will not be shown: "
    )

client = genai.Client()

MODEL_NAME = "gemini-flash-latest"

print("Model client ready")


Model client ready


In [6]:
approved_guidance = {
    "refund_policy": "Refund requests must be checked against the purchase date and product condition. A staff member must approve the final response.",
    "delivery_delay": "Delivery delay cases should include the order reference, expected delivery date and any courier update before a staff member contacts the customer.",
    "account_access": "Account access issues require identity checks before any account information is discussed or changed.",
}

service_request = """
A customer says their order has not arrived. They are frustrated because the delivery
date has passed, and they want someone to tell them what will happen next.
"""

print("Approved guidance topics:", ", ".join(approved_guidance.keys()))


Approved guidance topics: refund_policy, delivery_delay, account_access


In [7]:
def extract_json(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        print("The model did not return valid JSON. Check the raw response below, then rerun the cell.")
        print(text)
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError as e:
        print(f"Could not parse JSON: {e}\nRaw response: {text}")
        return {}


decision_prompt = f"""
You are helping route a service request inside a controlled AI system.

Allowed actions:
- retrieve_guidance
- human_review

Allowed guidance topics:
- refund_policy
- delivery_delay
- account_access
- unknown

Return only JSON using this exact structure:
{{
  "action": "retrieve_guidance or human_review",
  "topic": "refund_policy, delivery_delay, account_access or unknown",
  "confidence": 0.0,
  "reason": "short reason"
}}

Service request:
{service_request}
"""

decision_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=decision_prompt,
)

decision = extract_json(decision_response.text)
print(json.dumps(decision, indent=2))

{
  "action": "retrieve_guidance",
  "topic": "delivery_delay",
  "confidence": 0.95,
  "reason": "The customer is inquiring about an order that has not arrived past its delivery date."
}


In [9]:

def retrieve_guidance(topic):
    return approved_guidance.get(
        topic,
        "No approved guidance found. Route this request to a human reviewer.",
    )


state = {
    "request": service_request.strip(),
    "model_decision": decision,
    "tool_used": None,
    "tool_result": None,
    "status": "needs_human_review",
}

action = decision.get("action", "human_review")
topic = decision.get("topic", "unknown")

try:
    confidence = float(decision.get("confidence", 0))
except (TypeError, ValueError):
    confidence = 0

if (
    action == "retrieve_guidance"
    and topic in approved_guidance
    and confidence >= 0.65
):
    state["tool_used"] = "retrieve_guidance"
    state["tool_result"] = retrieve_guidance(topic)
else:
    state["tool_used"] = "none"
    state["tool_result"] = "Fallback: send to human review because the request was unclear or confidence was too low."

print(json.dumps(state, indent=2))


{
  "request": "A customer says their order has not arrived. They are frustrated because the delivery\ndate has passed, and they want someone to tell them what will happen next.",
  "model_decision": {
    "action": "retrieve_guidance",
    "topic": "delivery_delay",
    "confidence": 0.95,
    "reason": "The customer is inquiring about an order that has not arrived past its delivery date."
  },
  "tool_used": "retrieve_guidance",
  "tool_result": "Delivery delay cases should include the order reference, expected delivery date and any courier update before a staff member contacts the customer.",
  "status": "needs_human_review"
}
